# Deep Hedging d'un Put europeen

Notebook executable de bout en bout : simulation Black--Scholes sous probabilite reelle, entrainement de reseaux de couverture, comparaison au delta hedging et tests de robustesse. Les sorties sont volontairement nettoyees dans le depot ; relancez les cellules pour regenerer les tableaux et figures.

## Cadre

On vend un Put europeen de payoff $Z=(K-S_T)^+$ et on recoit une prime connue $p_0$. Pour une strategie previsible $\delta_k$ en actions, le P&L terminal est

$$\Pi_T = -Z + p_0 + \sum_{k=0}^{n-1} \delta_k(S_{k+1}-S_k) - C_T(\delta).$$

L'objectif principal est $\min_\theta E_P[\Pi_T(\theta)^2]$. Les trajectoires sont simulees sous la probabilite reelle $P$ ; la formule risque-neutre Black--Scholes sert seulement a fixer la prime de reference et le benchmark delta.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from model import build_hedger, default_model_configs
from simulation import MarketConfig, put_payoff, set_seed, simulate_gbm_paths
from training import TrainConfig, evaluate_delta_hedge, evaluate_strategy, hedging_pnl, train_deep_hedger
from utils import metrics_to_frame, plot_metric_bar, plot_pnl_distribution, plot_training_history

set_seed(2026)
torch.set_num_threads(2)
FIG_DIR = ROOT / "report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
market = MarketConfig(
    spot=100.0,
    strike=100.0,
    maturity=1.0,
    rate=0.0,
    drift=0.03,
    volatility=0.20,
    n_steps=30,
)
train_cfg = TrainConfig(
    n_epochs=10,
    n_train_paths=4096,
    n_test_paths=8192,
    batch_size=512,
    learning_rate=1e-3,
    seed=2026,
)
print(f"Prime Black--Scholes connue p0 = {market.option_premium:.4f}")

In [ ]:
paths = simulate_gbm_paths(market, 20_000, seed=11, antithetic=True)
payoffs = put_payoff(paths, market.strike).numpy()

plt.figure(figsize=(7, 4))
plt.plot(paths[:30].T, alpha=0.35)
plt.title("Exemples de trajectoires GBM sous P")
plt.xlabel("Date de couverture")
plt.ylabel("Sous-jacent")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "sample_paths.png", dpi=160)
plt.show()

pd.Series(payoffs).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

In [ ]:
trained = {}
histories = {}
metrics = [evaluate_delta_hedge(market, train_cfg.n_test_paths, seed=777)]

for name, model_cfg in default_model_configs().items():
    set_seed(2026)
    model = build_hedger(model_cfg)
    result = train_deep_hedger(model, market, train_cfg, progress=False)
    trained[name] = result["model"]
    histories[name] = result["history"]
    row = evaluate_strategy(result["model"], market, train_cfg.n_test_paths, seed=777)
    row["strategy"] = name
    metrics.append(row)

table = metrics_to_frame(metrics)
table

In [ ]:
plot_training_history(histories, FIG_DIR / "training_history.png")
plt.show()

plot_metric_bar(table, "mse", FIG_DIR / "architecture_mse.png")
plt.show()

In [ ]:
eval_paths = simulate_gbm_paths(market, train_cfg.n_test_paths, seed=888, antithetic=True)
payoff = put_payoff(eval_paths, market.strike)
pnl_by_strategy = {}

# Delta benchmark
from simulation import delta_hedge_positions
positions_delta = delta_hedge_positions(eval_paths, market)
pnl_by_strategy["Delta BS"] = hedging_pnl(positions_delta, eval_paths, payoff, market.option_premium).numpy()

for name, model in trained.items():
    with torch.no_grad():
        positions = model(eval_paths, market)
        pnl_by_strategy[name] = hedging_pnl(positions, eval_paths, payoff, market.option_premium).numpy()

plot_pnl_distribution(pnl_by_strategy, FIG_DIR / "pnl_distribution.png")
plt.show()

## Robustesse

On relance une version compacte de l'entrainement MLP profond pour differents strikes, volatilites, maturites et nombres de dates. Les resultats sont indicatifs mais hors-echantillon et calcules avec intervalles de confiance sur le MSE.

In [ ]:
robust_cfg = TrainConfig(n_epochs=5, n_train_paths=2048, n_test_paths=4096, batch_size=512, seed=123)
scenarios = [
    ("K=90", MarketConfig(strike=90, volatility=0.20, maturity=1.0, n_steps=30)),
    ("K=100", MarketConfig(strike=100, volatility=0.20, maturity=1.0, n_steps=30)),
    ("K=110", MarketConfig(strike=110, volatility=0.20, maturity=1.0, n_steps=30)),
    ("sigma=30%", MarketConfig(strike=100, volatility=0.30, maturity=1.0, n_steps=30)),
    ("T=0.5", MarketConfig(strike=100, volatility=0.20, maturity=0.5, n_steps=30)),
    ("n=12", MarketConfig(strike=100, volatility=0.20, maturity=1.0, n_steps=12)),
]
robust_rows = []
for label, scen_market in scenarios:
    set_seed(42)
    model = build_hedger(default_model_configs()["MLP profond"])
    train_deep_hedger(model, scen_market, robust_cfg, progress=False)
    deep_row = evaluate_strategy(model, scen_market, robust_cfg.n_test_paths, seed=999)
    delta_row = evaluate_delta_hedge(scen_market, robust_cfg.n_test_paths, seed=999)
    robust_rows.append({"scenario": label, "strategy": "MLP profond", **deep_row})
    robust_rows.append({"scenario": label, "strategy": "Delta BS", **delta_row})

robust_table = pd.DataFrame(robust_rows)[["scenario", "strategy", "mse", "rmse", "variance", "mse_ci_low", "mse_ci_high"]]
robust_table

## Extension : couts de transaction et CVaR

Le code permet d'ajouter des couts proportionnels $\lambda S_k |\Delta\delta_k|$ comme dans l'esprit de l'article. Une perte CVaR empirique est aussi fournie dans `training.py`; elle penalise davantage la queue gauche du P&L que la loss quadratique.

In [ ]:
cost = 0.001
cost_model = build_hedger(default_model_configs()["MLP profond"])
train_deep_hedger(cost_model, market, TrainConfig(n_epochs=6, n_train_paths=4096, n_test_paths=8192, batch_size=512, seed=314), proportional_cost=cost, progress=False)

cvar_model = build_hedger(default_model_configs()["MLP profond"])
train_deep_hedger(cvar_model, market, TrainConfig(n_epochs=6, n_train_paths=4096, n_test_paths=8192, batch_size=512, seed=315), loss_name="cvar", progress=False)

extension_table = metrics_to_frame([
    evaluate_delta_hedge(market, 8192, seed=1234, proportional_cost=cost),
    {**evaluate_strategy(cost_model, market, 8192, seed=1234, proportional_cost=cost), "strategy": "MLP profond + couts"},
    {**evaluate_strategy(cvar_model, market, 8192, seed=1234), "strategy": "MLP profond CVaR"},
])
extension_table